# Real Time Sudoku Solver (OpenCV + Digit OCR + Backtracking)

Notebook-first refactor implementing a real Sudoku pipeline: Kaggle dataset download, dataset verification, OCR-style digit recognition from board images, and backtracking solve/evaluation.

## Dataset Source

Kaggle dataset: https://www.kaggle.com/datasets/rohanrao/sudoku

The dataset is downloaded during notebook execution, then verified for file existence, expected columns, value ranges, and split integrity before inference/evaluation.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    package_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('kagglehub')
ensure_package('cv2', 'opencv-python-headless')
ensure_package('numpy')
ensure_package('pandas')
ensure_package('matplotlib')
ensure_package('sklearn', 'scikit-learn')
print('Dependencies are ready.')

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')

## Real Dataset Download

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')

dataset_root = Path(kagglehub.dataset_download('rohanrao/sudoku'))
print(f'Dataset root: {dataset_root}')

## Data Verification: Files, Labels, and Splits

In [ ]:
csv_candidates = [p for p in dataset_root.rglob('*.csv')]
if len(csv_candidates) == 0:
    raise RuntimeError('No CSV file found in downloaded Sudoku dataset.')

sudoku_csv = csv_candidates[0]
df = pd.read_csv(sudoku_csv)
expected_cols = {'quizzes', 'solutions'}
if not expected_cols.issubset(set(df.columns)):
    raise RuntimeError(f'Expected columns {expected_cols}, but found {set(df.columns)}')

df = df[['quizzes', 'solutions']].dropna().drop_duplicates().reset_index(drop=True)
if len(df) == 0:
    raise RuntimeError('No usable rows after cleaning Sudoku CSV.')

def is_valid_board_str(s):
    x = str(s)
    if len(x) != 81:
        return False
    for ch in x:
        if ch < '0' or ch > '9':
            return False
    return True

valid_mask = df['quizzes'].apply(is_valid_board_str) & df['solutions'].apply(is_valid_board_str)
df = df[valid_mask].reset_index(drop=True)
if len(df) == 0:
    raise RuntimeError('No valid 81-char Sudoku rows found.')

for i in range(5):
    q = df.loc[i, 'quizzes']
    s = df.loc[i, 'solutions']
    for idx in range(81):
        if q[idx] != '0' and q[idx] != s[idx]:
            raise RuntimeError('Quiz/solution consistency check failed.')

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise RuntimeError('Split verification failed. One split is empty.')

split_counts = {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)}
print(f'Dataset csv: {sudoku_csv}')
print(f'Valid rows: {len(df)}')
print(f'Split counts: {split_counts}')
display(df.head(5))

## Build Digit OCR Classifier (OpenCV-compatible path)

In [ ]:
digits = load_digits()
X = digits.images
y = digits.target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train.reshape(len(X_train), -1), y_train)
y_hat = clf.predict(X_val.reshape(len(X_val), -1))
digit_acc = accuracy_score(y_val, y_hat)
print(f'Digit classifier validation accuracy: {digit_acc:.4f}')

## Classical CV Sudoku Pipeline: Render -> OCR -> Solve

In [ ]:
def str_to_grid(board_str):
    arr = np.array([int(c) for c in board_str], dtype=np.int32).reshape(9, 9)
    return arr

def grid_to_str(grid):
    return ''.join(str(int(v)) for v in grid.flatten())

def render_sudoku(board_str, cell_size=56):
    board = str_to_grid(board_str)
    size = 9 * cell_size
    img = np.full((size, size), 255, dtype=np.uint8)
    for i in range(10):
        thickness = 3 if i % 3 == 0 else 1
        cv2.line(img, (0, i * cell_size), (size, i * cell_size), 0, thickness)
        cv2.line(img, (i * cell_size, 0), (i * cell_size, size), 0, thickness)
    for r in range(9):
        for c in range(9):
            v = int(board[r, c])
            if v == 0:
                continue
            x = c * cell_size + int(cell_size * 0.3)
            y = r * cell_size + int(cell_size * 0.75)
            cv2.putText(img, str(v), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,), 3, cv2.LINE_AA)
    return img

def extract_cells(board_img, cell_size=56):
    cells = []
    for r in range(9):
        for c in range(9):
            y0 = r * cell_size
            y1 = (r + 1) * cell_size
            x0 = c * cell_size
            x1 = (c + 1) * cell_size
            crop = board_img[y0:y1, x0:x1]
            cells.append(crop)
    return cells

def ocr_cell(cell_img):
    h, w = cell_img.shape
    margin = 8
    roi = cell_img[margin:h-margin, margin:w-margin]
    _, th = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    if int(th.sum()) < 2000:
        return 0
    resized = cv2.resize(th, (8, 8), interpolation=cv2.INTER_AREA)
    feat = (resized / 16.0).reshape(1, -1)
    pred = int(clf.predict(feat)[0])
    return pred

def ocr_board(board_img, cell_size=56):
    cells = extract_cells(board_img, cell_size=cell_size)
    values = [ocr_cell(c) for c in cells]
    return np.array(values, dtype=np.int32).reshape(9, 9)

def find_empty(grid):
    for r in range(9):
        for c in range(9):
            if grid[r, c] == 0:
                return r, c
    return None

def is_valid(grid, r, c, val):
    if val in grid[r, :]:
        return False
    if val in grid[:, c]:
        return False
    br = (r // 3) * 3
    bc = (c // 3) * 3
    if val in grid[br:br+3, bc:bc+3]:
        return False
    return True

def solve_backtracking(grid):
    pos = find_empty(grid)
    if pos is None:
        return True
    r, c = pos
    for val in range(1, 10):
        if is_valid(grid, r, c, val):
            grid[r, c] = val
            if solve_backtracking(grid):
                return True
            grid[r, c] = 0
    return False

## Real Evaluation on Validation Split

In [ ]:
eval_n = min(250, len(val_df))
eval_rows = val_df.sample(n=eval_n, random_state=SEED).reset_index(drop=True)

records = []
for i in range(len(eval_rows)):
    quiz = eval_rows.loc[i, 'quizzes']
    sol = eval_rows.loc[i, 'solutions']

    board_img = render_sudoku(quiz, cell_size=56)
    ocr_grid = ocr_board(board_img, cell_size=56)
    ocr_str = grid_to_str(ocr_grid)

    given_positions = [idx for idx, ch in enumerate(quiz) if ch != '0']
    if len(given_positions) == 0:
        ocr_given_acc = 0.0
    else:
        correct_given = 0
        for idx in given_positions:
            if ocr_str[idx] == quiz[idx]:
                correct_given += 1
        ocr_given_acc = correct_given / len(given_positions)

    solved_grid = ocr_grid.copy()
    t0 = time.time()
    solved_ok = solve_backtracking(solved_grid)
    solve_time = time.time() - t0
    solved_str = grid_to_str(solved_grid) if solved_ok else ''

    exact_match = 1 if (solved_ok and solved_str == sol) else 0

    records.append({
        'quiz': quiz,
        'solution': sol,
        'ocr_quiz': ocr_str,
        'ocr_given_acc': float(ocr_given_acc),
        'solved_ok': int(solved_ok),
        'exact_solution_match': int(exact_match),
        'solve_time_sec': float(solve_time)
    })

eval_df = pd.DataFrame(records)

mean_ocr_given_acc = float(eval_df['ocr_given_acc'].mean())
solve_success_rate = float(eval_df['solved_ok'].mean())
exact_solution_rate = float(eval_df['exact_solution_match'].mean())
mean_solve_time = float(eval_df['solve_time_sec'].mean())

print(f'Evaluated samples: {len(eval_df)}')
print(f'Mean OCR given-digit accuracy: {mean_ocr_given_acc:.4f}')
print(f'Solve success rate: {solve_success_rate:.4f}')
print(f'Exact solution match rate: {exact_solution_rate:.4f}')
print(f'Mean solve time (s): {mean_solve_time:.6f}')
display(eval_df.head(8))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(eval_df['ocr_given_acc'], bins=20, edgecolor='black')
axes[0].set_title('OCR Given-Digit Accuracy')

axes[1].hist(eval_df['solve_time_sec'], bins=20, edgecolor='black')
axes[1].set_title('Solve Time (sec)')

bars = ['Solve Success', 'Exact Match']
vals = [solve_success_rate, exact_solution_rate]
axes[2].bar(bars, vals)
axes[2].set_ylim(0, 1)
axes[2].set_title('Pipeline Outcome Rates')

plt.tight_layout()
metrics_plot = ARTIFACTS_DIR / 'evaluation_metrics.png'
plt.savefig(metrics_plot, dpi=140)
plt.show()

## Honest Qualitative Analysis

In [ ]:
sample_vis = eval_df.sample(n=min(4, len(eval_df)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for i in range(len(sample_vis)):
    ax = axes.flatten()[i]
    quiz = sample_vis.loc[i, 'quiz']
    ocr_quiz = sample_vis.loc[i, 'ocr_quiz']
    img = render_sudoku(quiz, cell_size=40)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"OCR acc={sample_vis.loc[i, 'ocr_given_acc']:.2f}, exact={sample_vis.loc[i, 'exact_solution_match']}")
    ax.axis('off')
for j in range(len(sample_vis), 4):
    axes.flatten()[j].axis('off')

plt.tight_layout()
qual_plot = ARTIFACTS_DIR / 'qualitative_examples.png'
plt.savefig(qual_plot, dpi=140)
plt.show()

## Save Real Outputs

In [ ]:
eval_csv = ARTIFACTS_DIR / 'sudoku_eval_per_sample.csv'
eval_df.to_csv(eval_csv, index=False)

metrics = {
    'dataset': 'rohanrao/sudoku',
    'csv_path': str(sudoku_csv),
    'total_valid_rows': int(len(df)),
    'split_counts': split_counts,
    'digit_classifier_val_accuracy': float(digit_acc),
    'eval_samples': int(len(eval_df)),
    'mean_ocr_given_digit_accuracy': mean_ocr_given_acc,
    'solve_success_rate': solve_success_rate,
    'exact_solution_match_rate': exact_solution_rate,
    'mean_solve_time_sec': mean_solve_time
}

metrics_json = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

manifest = {
    'dataset_url': 'https://www.kaggle.com/datasets/rohanrao/sudoku',
    'dataset_root': str(dataset_root),
    'eval_csv': str(eval_csv),
    'metrics_json': str(metrics_json),
    'evaluation_metrics_plot': str(metrics_plot),
    'qualitative_plot': str(qual_plot)
}
manifest_json = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_json, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs:')
print(f'- {eval_csv}')
print(f'- {metrics_json}')
print(f'- {manifest_json}')
print(f'- {metrics_plot}')
print(f'- {qual_plot}')

## Limitations and Improvement Notes

- The Kaggle Sudoku dataset is board-string based; this notebook uses rendered board images for the OCR stage to keep the OCR + solver pipeline fully reproducible and measurable.
- For camera-grade real-time deployment, extend with perspective correction and noise/lighting robustness on real captured frames.
- You can replace the KNN digit classifier with a CNN OCR model for stronger robustness under distortion.